In [0]:
silver_patients=spark.read.table("workspace.silver.patients")
silver_encounters=spark.read.table("workspace.silver.encounters")
silver_conditions=spark.read.table("workspace.silver.conditions")
silver_observations=spark.read.table("workspace.silver.observations")
silver_medications=spark.read.table("workspace.silver.medications")
silver_claims=spark.read.table("workspace.silver.claims")

In [0]:
#display(silver_patients)

DASHBOARD 1

In [0]:
#display(silver_encounters)

In [0]:
from pyspark.sql.functions import col, when, year, current_date, datediff

dim_patient = silver_patients.select(
    col("patient_id").alias("patient_key"),
    col("gender"),
    col("dob"),
    (datediff(current_date(), col("dob")) / 365.25).cast("int").alias("age"),
    col("city"),
    col("state"),
    when(col("deceased_datetime").isNotNull(), 1).otherwise(0).alias("is_deceased"),
    when((datediff(current_date(), col("dob")) / 365.25) < 18, "Child")
        .when((datediff(current_date(), col("dob")) / 365.25) < 65, "Adult")
        .otherwise("Senior").alias("age_group")
).dropDuplicates(["patient_key"])

#display(dim_patient)

In [0]:
#display(silver_conditions)

In [0]:
dim_condition = (
    silver_conditions
    .select(
        col("condition_code").alias("condition_key"),
        col("condition_code"),
        col("condition_description")
    )
    .dropDuplicates(["condition_key"])
)

#display(dim_condition)

In [0]:
from pyspark.sql.functions import col, year, month, date_format, quarter

dim_date = (
    silver_encounters
    .select(
        date_format(col("encounter_start"), "yyyyMMdd").cast("int").alias("date_key"),
        col("encounter_start").alias("date"),
        year(col("encounter_start")).alias("year"),
        month(col("encounter_start")).alias("month"),
        date_format(col("encounter_start"), "MMMM").alias("month_name"),
        quarter(col("encounter_start")).alias("quarter")
    )
    .dropDuplicates(["date_key"])
)

#display(dim_date)

In [0]:
from pyspark.sql.functions import col, date_format, lit,countDistinct

fact_encounter = (
    silver_encounters
    .select(
        col("encounter_id").alias("encounter_key"),
        col("patient_id").alias("patient_key"),
        date_format(col("encounter_start"), "yyyyMMdd").cast("int").alias("date_key"),
        col("encounter_class").alias("encounter_type"),
        lit(1).alias("encounter_count")
    )
    .dropDuplicates(["encounter_key"])
)

#display(fact_encounter)

In [0]:
fact_patient_conditions = (
    silver_conditions
    .select(
        col("patient_id").alias("patient_key"),
        col("condition_code").alias("condition_key"),
        date_format(col("condition_onset"), "yyyyMMdd").cast("int").alias("date_key"),
        col("clinical_status"),
        lit(1).alias("condition_count")
    )
)
#display(fact_patient_conditions)

In [0]:
fact_claims = (
    silver_claims
    .select(
        col("claim_id").alias("claim_key"),
        col("patient_id").alias("patient_key"),
        col("encounter_id").alias("encounter_key"),
        col("provider_id").alias("provider_key"),
        col("insurance_name"),
        col("claim_status"),
        col("total").cast("double").alias("claim_amount"),
        col("priority_code"),
        lit(1).alias("claim_count")
    )
    .dropDuplicates(["claim_key"]
    )
)


In [0]:
#For Analytics Table

In [0]:
gold_population_health = (
    fact_encounter.alias("f")
    .join(dim_patient.alias("p"), col("f.patient_key") == col("p.patient_key"), "left")
    .join(dim_date.alias("d"), col("f.date_key") == col("d.date_key"), "left")
    .select(
        col("f.encounter_key"),
        col("f.patient_key"),
        col("f.date_key"),
        col("f.encounter_type"),
        col("f.encounter_count"),
        col("p.gender"),
        col("p.age"),
        col("p.age_group"),
        col("p.city"),
        col("p.state"),
        col("p.is_deceased"),
        col("d.date"),
        col("d.year"),
        col("d.month"),
        col("d.month_name"),
        col("d.quarter")
    )
)

In [0]:
gold_clinical_analytics = (fact_patient_conditions.alias("f")
    .join(dim_patient.alias("p"), col("f.patient_key") == col("p.patient_key"), "left")
    .join(dim_condition.alias("c"), col("f.condition_key") == col("c.condition_key"), "left")
    .join(dim_date.alias("d"), col("f.date_key") == col("d.date_key"), "left")
    .select(
        col("f.patient_key"),
        col("f.condition_key"),
        col("f.date_key"),
        col("f.clinical_status"),
        col("f.condition_count"),
        col("p.gender"),
        col("p.age"),
        col("p.age_group"),
        col("p.city"),
        col("p.state"),
        col("p.is_deceased"),
        col("c.condition_description"),
        col("d.year"),
        col("d.month_name"),
        col("d.quarter")
    )
)

In [0]:
gold_patient_risk = (
    fact_patient_conditions.groupBy("patient_key")
    .agg(
        countDistinct("condition_key").alias("total_conditions")
    ).withColumn("risk_level",
                 when(col("total_conditions") >=10, "High")
                 .when(col("total_conditions") >=5, "Medium")
                 .when(col("total_conditions") >=1, "Low")
                 .otherwise("Unknown")
        )
)
#display(gold_patient_risk)

In [0]:
gold_revenue_cycle_analytics = (
    fact_claims.alias("f")
    .join(
        dim_patient.alias("p"),
        col("f.patient_key") == col("p.patient_key"),
        "left"
    )
    .select(
        col("f.claim_key"),
        col("f.patient_key"),
        col("f.encounter_key"),
        col("f.provider_key"),
        col("f.insurance_name"),
        col("f.claim_status"),
        col("f.claim_amount"),
        col("f.priority_code"),
        col("f.claim_count"),
        col("p.gender"),
        col("p.age"),
        col("p.age_group"),
        col("p.city"),
        col("p.state"),
        col("p.is_deceased"),
        when(col("f.claim_amount") >= 1000, lit("High"))
            .when(col("f.claim_amount") >= 500, lit("Medium"))
            .otherwise(lit("Low")).alias("claim_value_band")
    )
)

display(gold_revenue_cycle_analytics.limit(5))

In [0]:
from pyspark.sql.functions import col, countDistinct
condition_pairs = (
    fact_patient_conditions.alias("a")
    .join(
        fact_patient_conditions.alias("b"),
        col("a.patient_key") == col("b.patient_key")
    )
    .filter(
        col("a.condition_key") < col("b.condition_key")
    )
)
condition_pairs = (
    condition_pairs
    .join(
        dim_condition.alias("c1"),
        col("a.condition_key") == col("c1.condition_key")
    )
    .join(
        dim_condition.alias("c2"),
        col("b.condition_key") == col("c2.condition_key")
    )
)
condition_cooccurrence = (
    condition_pairs
    .groupBy(
        col("c1.condition_description").alias("condition_a"),
        col("c2.condition_description").alias("condition_b")
    )
    .agg(
        countDistinct(col("a.patient_key")).alias("patient_count")
    )
    .orderBy(col("patient_count").desc())
)
#display(condition_cooccurrence)

In [0]:
dim_patient.write.format("delta").mode("overwrite").saveAsTable("gold.dim_patient")
dim_condition.write.format("delta").mode("overwrite").saveAsTable("gold.dim_condition")
dim_date.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.dim_date")
fact_encounter.write.format("delta").mode("overwrite").saveAsTable("gold.fact_encounter")
fact_patient_conditions.write.format("delta").mode("overwrite").saveAsTable("gold.fact_patient_conditions")
gold_population_health.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.gold_population_health")
gold_clinical_analytics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.gold_clinical_analytics")
gold_patient_risk.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.gold_patient_risk")
condition_cooccurrence.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.condition_cooccurrence")
gold_revenue_cycle_analytics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.gold_revenue_cycle_analytics")
fact_claims.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("gold.fact_claims")

In [0]:
# %sql
# SELECT
#     condition_description,
#     COUNT(*) AS diagnoses
# FROM gold.gold_clinical_analytics
# GROUP BY condition_description
# ORDER BY diagnoses DESC
# LIMIT 10

In [0]:
# %sql
# SELECT
#     clinical_status,
#     COUNT(*) AS diagnoses
# FROM gold.gold_clinical_analytics
# GROUP BY clinical_status
# ORDER BY diagnoses DESC

In [0]:
# %sql
# SELECT
#     patient_key,
#     COUNT(DISTINCT condition_key) AS condition_count
# FROM gold.gold_clinical_analytics
# GROUP BY patient_key
# ORDER BY condition_count DESC

In [0]:
# %sql
# SELECT
#     insurance_name,
#     COUNT(*) AS claims,
#     ROUND(SUM(claim_amount),2) AS total_claim_amount
# FROM gold.gold_revenue_cycle_analytics
# GROUP BY insurance_name
# ORDER BY total_claim_amount DESC

In [0]:
# %sql
# SELECT
#     claim_status,
#     COUNT(*) AS claims
# FROM gold.gold_revenue_cycle_analytics
# GROUP BY claim_status
# ORDER BY claims DESC